In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random

%matplotlib inline

In [ ]:
root = Path('..')
data_path = root / 'data' / 'names.txt'
words = open(data_path).read().splitlines()

print(f'Words count: {len(words)}')

# build vocabulary of characters and mappings to/from ints

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i, s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

# build dataset split into train / val / test

def build_dataset(words, block_size=3):
    X, Y = [], []

    for w in words:

        context = [0] * block_size
        for ch in w + '.':
            ix = stoi[ch]
            X.append(context)
            Y.append(ix)
            context = context[1:] + [ix]
    
    X = torch.tensor(X)
    Y = torch.tensor(Y)
    print(X.shape, X.dtype, Y.shape, Y.dtype)
    return X, Y

X, Y = build_dataset(words)

s1 = int(0.8 * len(X))
s2 = int(0.9 * len(X))

X_train, Y_train = X[:s1], Y[:s1]
X_val, Y_val = X[s1:s2], Y[s1:s2]
X_test, Y_test = X[s2:], Y[s2:]

print(f'train: {len(X_train)} val: {len(X_val)} test: {len(X_test)}')

In [ ]:
# utility function for comparing the manual and torch gradients
def cmp(s, dt, t):
    """
    s: name of the variable
    dt: manual gradient
    t: torch gradient
    """

    ex = torch.all(dt == t.grad).item()
    app = torch.allclose(dt, t.grad)
    maxdiff = (dt - t.grad).abs().max().item()
    print(f'{s:15s} | exact: {str(ex):5s} | approx: {str(app):5s} | maxdiff: {maxdiff}')

In [ ]:
N_EMBD = 10
N_HIDDEN = 64
VOCAB_SIZE = len(stoi)
BLOCK_SIZE = 3
EPOCHS = 1_000
BATCH_SIZE = 32
LR = 0.135

g = torch.Generator().manual_seed(42)
C = torch.randn((VOCAB_SIZE, N_EMBD), generator=g)

# Layer 1
W1 = torch.randn((N_EMBD * BLOCK_SIZE, N_HIDDEN), generator=g) * (5/3) / ((N_EMBD * BLOCK_SIZE) ** 0.5)
b1 = torch.randn(N_HIDDEN, generator=g) * 0.1

# Layer 2
W2 = torch.randn((N_HIDDEN, VOCAB_SIZE), generator=g) * 0.1
b2 = torch.randn(VOCAB_SIZE, generator=g) * 0.1

# BatchNorm params
bngain = torch.ones((1, N_HIDDEN)) * 0.1 + 1.0
bnbias = torch.zeros((1, N_HIDDEN)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

total_params = sum(p.nelement() for p in parameters) # total number of parameters
print(f'Total parameters: {total_params}')

for p in parameters: p.requires_grad = True # enable gradients

In [ ]:
# constuct a minibatch
ix = torch.randint(0, X_train.shape[0], (BATCH_SIZE, ), generator=g)
Xb, Yb = X_train[ix], Y_train[ix]
print(Xb.shape, Yb.shape)

In [ ]:
# Forward propagation, divided into smaller parts, se we can compute backpropagation manually for every step

# Step 1: Embedding
emb = C[Xb] # embed the input character
embcat = emb.view(emb.shape[0], -1) # concatenate embeddings

# Step 2: Linear Layer 1
hprebn = embcat @ W1 + b1 # pre-batchnorm pre-activation hidden layer

# Step 3: BatchNorm
bnmeani = 1 / BATCH_SIZE * hprebn.sum(dim=0, keepdim=True)
bndiff_1 = hprebn - bnmeani
bndiff_2 = bndiff_1 ** 2
bnvar = 1 / (BATCH_SIZE - 1) * (bndiff_2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n - 1, not n)
bnvar_inv = (bnvar + 1e-5) ** -0.5
bnraw = bndiff_1 * bnvar_inv
hpreact = bngain * bnraw + bnbias

# Step 4: Non-linearity (tanh)
h = torch.tanh(hpreact) # hidden layer

# Step 5: Linear Layer 2
logits = h @ W2 + b2 # output layer

# Step 6: Cross-entropy loss (divided into smaller parts)
logit_maxes = logits.max(1, keepdim=True).values
norm_logits = logits - logit_maxes # subtract max for numerical stability
counts = norm_logits.exp()
counts_sum = counts.sum(1, keepdim=True)
counts_sum_inv = counts_sum ** -1
probs = counts * counts_sum_inv
logprobs = probs.log()
loss = -logprobs[range(BATCH_SIZE), Yb].mean()

# Step 7: PyTorch backward pass and retaining gradients for manual comparison
for p in parameters: p.grad = None # zero gradients

steps = [
    logprobs, probs, counts, counts_sum, counts_sum_inv,
    norm_logits, logit_maxes, logits, h, hpreact, bnraw,
    bnvar_inv, bnvar, bndiff_2, bndiff_1, bnmeani, hprebn,
    embcat, emb
] # order of steps in forward pass (from bottom to top)

for step in steps: step.retain_grad() # keep gradients for all steps

loss.backward()

print(f'loss: {loss.item()}')

print(logits.shape, logit_maxes.shape)

In [ ]:
# Exercice 1 : Manually backprop through the whole graph step by step

dlogprobs = torch.zeros_like(logprobs)
dlogprobs[range(BATCH_SIZE), Yb] = -1.0 / BATCH_SIZE
print(f'\ndlogprobs shape: {dlogprobs.shape}')
cmp('logprobs', dlogprobs, logprobs)

dprobs = 1 / probs * dlogprobs
print(f'\ndprobs shape: {dprobs.shape}')
cmp('probs', dprobs, probs)

dcounts_sum_inv = (counts * dprobs).sum(1, keepdim=True)
print(f'\ndcounts_sum_inv shape: {dcounts_sum_inv.shape}')
cmp('counts_sum_inv', dcounts_sum_inv, counts_sum_inv)

dcounts_sum = -1 * (counts_sum ** -2) * dcounts_sum_inv
print(f'\ndcounts_sum shape: {dcounts_sum.shape}')
cmp('counts_sum', dcounts_sum, counts_sum)

dcounts = counts_sum_inv * dprobs
dcounts += torch.ones_like(counts) * dcounts_sum
print(f'\ndcounts shape: {dcounts.shape}')
cmp('count', dcounts, counts)

dnorm_logits = counts * dcounts
print(f'\ndnorm_logits shape: {dnorm_logits.shape}')
cmp('norm_logits', dnorm_logits, norm_logits)

dlogits = dnorm_logits.clone()
dlogit_maxes = (dnorm_logits * -1).sum(1, keepdim=True)
print(f'\ndlogit_maxes shape: {dlogit_maxes.shape}')
cmp('logit_maxes', dlogit_maxes, logit_maxes)

dlogits += F.one_hot(logits.max(1).indices, num_classes=logits.shape[1]) * dlogit_maxes
print(f'\ndlogits shape: {dlogits.shape}')
cmp('logits', dlogits, logits)

dh = dlogits @ W2.T
dW2 = h.T @ dlogits
db2 = dlogits.sum(0)
print(f'\ndh shape: {dh.shape}, dW2 shape: {dW2.shape}, db2 shape: {db2.shape}')
cmp('h', dh, h)
cmp('W2', dW2, W2)
cmp('b2', db2, b2)

dhpreact = (1.0 - torch.tanh(hpreact) ** 2) * dh
print(f'\ndhpreact shape: {dhpreact.shape}')
cmp('hpreact', dhpreact, hpreact)

dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
dbnraw = bngain * dhpreact
dbnbias = (1 * dhpreact).sum(0, keepdim=True)
print(f'\ndbngain shape: {dbngain.shape}, dbnraw shape: {dbnraw.shape}, dbnbias shape: {dbnbias.shape}')
cmp('bngain', dbngain, bngain)
cmp('bnraw', dbnraw, bnraw)
cmp('bnbias', dbnbias, bnbias)

dbnvar_inv = (dbnraw * bndiff_1).sum(0, keepdim=True)
print(f'\ndbnvar_inv shape: {dbnvar_inv.shape}')
cmp('bnvar_inv', dbnvar_inv, bnvar_inv)

dbnvar = -0.5 * (bnvar + 1e-5) ** -1.5 * dbnvar_inv
print(f'\ndbnvar shape: {dbnvar.shape}')
cmp('bnvar', dbnvar, bnvar)

dbndiff_2 = (1.0 / (BATCH_SIZE - 1)) * torch.ones_like(bndiff_2) * dbnvar
print(f'\ndbndiff_2 shape: {dbndiff_2.shape}')
cmp('bndiff_2', dbndiff_2, bndiff_2)

# A lot of branching happening here, complete all steps
dbndiff_1 = bnvar_inv * dbnraw # 1st branch
dbndiff_1 += (2 * bndiff_1) * dbndiff_2 # 2nd branch
print(f'\ndbndiff_1 shape: {dbndiff_1.shape}')
cmp('bndiff_1', dbndiff_1, bndiff_1)

# bndiff_1 = hprebn - bnmeani
dhprebn = dbndiff_1.clone() # 1st branch
dbnmeani = (-torch.ones_like(bndiff_1) * dbndiff_1).sum(0, keepdim=True)
print(f'\ndbnmeani shape: {dbnmeani.shape}')
cmp('bnmeani', dbnmeani, bnmeani)

dhprebn += 1 / BATCH_SIZE * dbnmeani
print(f'\ndhprebn shape: {dhprebn.shape}')
cmp('hprebn', dhprebn, hprebn)

dembcat = dhprebn @ W1.T
dW1 = embcat.T @ dhprebn
db1 = dhprebn.sum(0)
print(f'\ndembcat shape: {dembcat.shape}, dW1 shape: {dW1.shape}, db1 shape: {db1.shape}')
cmp('embcat', dembcat, embcat)
cmp('W1', dW1, W1)
cmp('b1', db1, b1)

demp = dembcat.view(emb.shape)
print(f'\ndemp shape: {demp.shape}')
cmp('emp', demp, emb)

dC = torch.zeros_like(C)
for k in range(Xb.shape[0]):
    for j in range(Xb.shape[1]):
        ix = Xb[k, j]
        dC[ix] += demp[k, j]
print(f'\n dC shape: {dC.shape}')
cmp('C', dC, C)

In [ ]:
# Exercise 2 : Backprop through cross_entropy (simplify the expression)

loss_torch = F.cross_entropy(logits, Yb)
print(f'{loss_torch.item()}, diff: {(loss_torch - loss).item():.10f}')

dlogits = (F.softmax(logits, 1) - F.one_hot(Yb, num_classes=logits.size(1))) / BATCH_SIZE

cmp('dlogits', dlogits, logits)

plt.figure(figsize=(10, 10))
plt.imshow(dlogits.detach(), cmap='gray')

In [ ]:
# Exercise 3 : Backprop through batchnorm (simplify the expression)

dhprebn = bngain * bnvar_inv / BATCH_SIZE * \
    (BATCH_SIZE * dhpreact - dhpreact.sum(0) - BATCH_SIZE / (BATCH_SIZE - 1) \
    * bnraw * (dhpreact * bnraw).sum(0))

cmp('hprebn', dhprebn, hprebn)

In [ ]:
# Exercise 4 : Putting it all together

# Initialization

N_EMBD = 10
N_HIDDEN = 200
VOCAB_SIZE = len(stoi)
BLOCK_SIZE = 3
EPOCHS = 200_000
BATCH_SIZE = 32
LR = 0.1
LOSSI, STEPI = [], []

g = torch.Generator().manual_seed(42)
C = torch.randn((VOCAB_SIZE, N_EMBD), generator=g)

# Layer 1
W1 = torch.randn((N_EMBD * BLOCK_SIZE, N_HIDDEN), generator=g) * (5/3) / ((N_EMBD * BLOCK_SIZE) ** 0.5)
b1 = torch.randn(N_HIDDEN, generator=g) * 0.1

# Layer 2
W2 = torch.randn((N_HIDDEN, VOCAB_SIZE), generator=g) * 0.1
b2 = torch.randn(VOCAB_SIZE, generator=g) * 0.1

# BatchNorm params
bngain = torch.ones((1, N_HIDDEN)) * 0.1 + 1.0
bnbias = torch.zeros((1, N_HIDDEN)) * 0.1

parameters = [C, W1, b1, W2, b2, bngain, bnbias]

total_params = sum(p.nelement() for p in parameters) # total number of parameters
print(f'Total parameters: {total_params}')

for p in parameters: p.requires_grad = True # enable gradients

with torch.no_grad():

    for epoch in range(EPOCHS):
        # Constuct a minibatch
        ix = torch.randint(0, X_train.shape[0], (BATCH_SIZE, ), generator=g)
        Xb, Yb = X_train[ix], Y_train[ix]

        # Forward pass:
        # Step 1: Embedding
        emb = C[Xb] # embed the input character
        embcat = emb.view(emb.shape[0], -1) # concatenate embeddings

        # Step 2: Linear Layer 1
        hprebn = embcat @ W1 + b1 # pre-batchnorm pre-activation hidden layer

        # Step 3: BatchNorm
        bnmeani = 1 / BATCH_SIZE * hprebn.sum(dim=0, keepdim=True)
        bndiff_1 = hprebn - bnmeani
        bndiff_2 = bndiff_1 ** 2
        bnvar = 1 / (BATCH_SIZE - 1) * (bndiff_2).sum(0, keepdim=True) # note: Bessel's correction (dividing by n - 1, not n)
        bnvar_inv = (bnvar + 1e-5) ** -0.5
        bnraw = bndiff_1 * bnvar_inv
        hpreact = bngain * bnraw + bnbias

        # Step 4: Non-linearity (tanh)
        h = torch.tanh(hpreact) # hidden layer

        # Step 5: Linear Layer 2
        logits = h @ W2 + b2 # output layer

        # Step 6: Cross-entropy loss (divided into smaller parts)
        logit_maxes = logits.max(1, keepdim=True).values
        norm_logits = logits - logit_maxes # subtract max for numerical stability
        counts = norm_logits.exp()
        counts_sum = counts.sum(1, keepdim=True)
        counts_sum_inv = counts_sum ** -1
        probs = counts * counts_sum_inv
        logprobs = probs.log()
        loss = -logprobs[range(BATCH_SIZE), Yb].mean()

        # Backward pass:
        for p in parameters: p.grad = None
        # loss.backward() # correctness comparison check

        # Step 1: Cross-entropy loss gradient
        dlogits = (F.softmax(logits, 1) - F.one_hot(Yb, num_classes=logits.size(1))) / BATCH_SIZE

        # Step 2: Linear Layer 2 backprop
        dh = dlogits @ W2.T
        dW2 = h.T @ dlogits
        db2 = dlogits.sum(0)

        # Step 3: Tanh backprop
        dhpreact = (1.0 - h**2) * dh

        # Step 4: BatchNorm backprop
        dbngain = (bnraw * dhpreact).sum(0, keepdim=True)
        dbnbias = dhpreact.sum(0, keepdim=True)
        dhprebn = bngain * bnvar_inv / BATCH_SIZE * \
            (BATCH_SIZE * dhpreact - dhpreact.sum(0) - BATCH_SIZE / (BATCH_SIZE - 1) \
            * bnraw * (dhpreact * bnraw).sum(0))

        # Step 5: Linear Layer 1 backprop
        dembcat = dhprebn @ W1.T
        dW1 = embcat.T @ dhprebn
        db1 = dhprebn.sum(0)

        # Step 6: Embedding backprop
        demb = dembcat.view(emb.shape)
        dC = torch.zeros_like(C)
        for k in range(Xb.shape[0]):
            for j in range(Xb.shape[1]):
                ix = Xb[k,j]
                dC[ix] += demb[k,j]
        gradients = [dC, dW1, db1, dW2, db2, dbngain, dbnbias]

        # Step 7: Update
        if epoch < 100000: LR = 0.01
        for p, grad in zip(parameters, gradients):
            p.data += -LR * grad

        # Track stats
        if epoch % 1000 == 0:
            print(f'epoch: {epoch:10d}/{EPOCHS:1d} | loss: {loss.item():.3f}')
            LOSSI.append(loss.log10().item())
            STEPI.append(epoch)

# Gradient comparison: manual vs. PyTorch
# 1. Comment the 'with torch.no_grad():' context manager around training loop
# 2. Uncomment 'loss.backward()' before calculating manual gradients
# for p, g in zip(parameters, gradients):
#     print(str(tuple(p.shape)))
#     cmp(str(tuple(p.shape)), g, p)

plt.plot(STEPI, LOSSI)

In [ ]:
# BatchNorm calibration after training

with torch.no_grad():

    # Pass the training set through
    emb = C[X_train]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1

    # Measure the mean/std over the entire training set
    bnmean = hpreact.mean(0, keepdim=True)
    bnvar = hpreact.var(0, keepdim=True, unbiased=True)

In [ ]:
# evaluation

@torch.no_grad()
def split_loss(split):
    x,y = {
        'train': (X_train, Y_train),
        'val': (X_val, Y_val),
        'test': (X_val, Y_val),
    }[split]
    emb = C[x]
    embcat = emb.view(emb.shape[0], -1)
    hpreact = embcat @ W1 + b1
    hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
    h = torch.tanh(hpreact)
    logits = h @ W2 + b2
    loss = F.cross_entropy(logits, y)
    print(split, loss.item())

split_loss('train')
split_loss('val')

In [ ]:
# sample from the model

g = torch.Generator().manual_seed(42)

for _ in range(20):

    out = []
    context = [0] * BLOCK_SIZE

    while True:

        emb = C[torch.tensor([context])]    
        embcat = emb.view(emb.shape[0], -1)
        hpreact = embcat @ W1 + b1
        hpreact = bngain * (hpreact - bnmean) * (bnvar + 1e-5)**-0.5 + bnbias
        h = torch.tanh(hpreact)
        logits = h @ W2 + b2

        probs = F.softmax(logits, dim=1)
        ix = torch.multinomial(probs, num_samples=1, generator=g).item()
        context = context[1:] + [ix]
        out.append(ix)
        if ix == 0:
            break

    print(''.join(itos[i] for i in out))